In [ ]:
!pip install gradio pymupdf scikit-learn gdown requests --quiet
# ==========================================
# gradio            : UI liberary that allows to turn the code into a HTML like page (same structure that we will use with html that allows css and html combination with table)
# pymupdf           : liberary known as fitz allows to open PDF files and extract text from them while keeping the structure and location(page) of each sentence
# scikit-learn      : machine learning Liberary - used for TF-IDF aswell as Cosine Similarity
# TF-IDF            : builds dictionary(creates vectoric representaion that allows Cosine Similarity between questions to data sources) from articles and allows to search words -transfers Data From articles to the rest of the modele- the Retriever of the RAG
# Cosine Similarity : allows for mathematical calculations to detirmine how close the our question is to the given Texts(articles) - second part of the Retriever of the RAG
# gdown             : Liberary that we use to bypass Google Drives Defences in order to download files from it for IDES like Colab(we keep the articles on our google Drive to be used at the begining of each run)
# requests          : Gives the Code API to communicate and make http requests to Gemini servers. -- the Generator of RAG
# --quite           : cosmetic - Dont Show the texts that will be printed from intalling the Libereris
# ==========================================

import os
import re
import requests
import gradio as gr
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


gr.close_all()# closeing all Previosly opened interfaces to clear memory -- prevents Port colitions

# ==========================================
#  GEMINI has a few possible version and sometimes some of them are unavailable which makes the program crush
#  so we try each on and if its unavailable we downgrade to the next possibility (Code Robustness)
#  1. handshake with Googles Gemini
#  2. sends the key to recive allowed Ai modeles
#  3. Filters to the modeles that support generateContent(the Generator)
#  4. prioritises FlashModels after that pro (stronger) models and uses the best possible option
# ==========================================
API_KEY = "ENTER API KEY HERE"
ACTIVE_MODEL = "models/gemini-1.5-flash" #default value will change in discovery_model if its unavailable

In [ ]:
def discovery_model():#Fallback Mechanism
#{
    global ACTIVE_MODEL
    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models?key={API_KEY}"
        res = requests.get(url, timeout=10)#get the available models for my api key , 10 sec wait time to avoid deadlocks

        if res.status_code == 200:#200 == success
            models = res.json().get('models', [])# get the answers in a json dictionary type
            valid_models = [m['name'] for m in models if 'generateContent' in m.get('supportedGenerationMethods', [])] #check if the recived model can work with generateContent (can answer questions)

            if valid_models:#priritizing models
                flash = [m for m in valid_models if 'flash' in m] # does the model contain the word flash? if so add the model to flash list
                pro = [m for m in valid_models if 'pro' in m]#does the model contain the word pro?if so add the model to pro list

                if flash: ACTIVE_MODEL = flash[0]#prioritize flash
                elif pro: ACTIVE_MODEL = pro[0]
                else: ACTIVE_MODEL = valid_models[0]

                print(f"Active Model: {ACTIVE_MODEL}")
    except Exception:
        pass
#}
#discovery_model()

In [ ]:


def call_gemini_direct(prompt):
  # ==========================================
  # This is the generator:
  # send the datato the Ai model
  # ==========================================
  #{
    url = f"https://generativelanguage.googleapis.com/v1beta/{ACTIVE_MODEL}:generateContent?key={API_KEY}"#here we ask for a specific model instead of a list like before(Model Invocation)
    headers = {'Content-Type': 'application/json'}#the is the type of format the data will be sent to the AI API
    data = {"contents": [{"parts": [{"text": prompt}]}]}#the needed format to be sent to the AI Generator - json Frame

    response = requests.post(url, headers=headers, json=data, timeout=15)# send the data to the Ai Model wait for up to 15 sec
    response.raise_for_status() #catch errors
    return response.json()['candidates'][0]['content']['parts'][0]['text']#this is the data we get back from the Ai model, notice we do not need to use asyncronic cmds here
#}
#call_gemini_direct(prompt)



In [ ]:

#Global variables for use to save data from articles
rag_chunks = []
vectorizer = None
tfidf_matrix = None

In [ ]:
def clean_text(text):
  #{
    text = text.replace("-", " ").replace("\n", " ")
    return re.sub(r'\s+', ' ', text).strip()
  #}
  #cleat_text

In [ ]:

def chunk_text(text, chunk_size=200, overlap=50):
  #{
    words = text.split() # split text into words
    chunks = []

    # step determines how much we jump forward.
    # chunk_size - overlap means we go back 50 words each time to avoid cutting sentences!
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.strip()) > 50: # if we have less than 50 Characters left just enter all of then into the current chunk - avoid short chunks
            chunks.append(chunk)

    return chunks
  #}
#chunk_text



In [ ]:
# ==========================================
# this part turns the pdf files into database
# ==========================================
def initialize_rag_system():
  #{
    global rag_chunks, vectorizer, tfidf_matrix
    rag_chunks = []
    base_download_path = "./papers"
    os.makedirs(base_download_path, exist_ok=True)#location where data will be saved, checks if path exist
    folder_url = "https://drive.google.com/drive/folders/1W_9CDVLOetS7hD-tjC7DoiEB-VvmrPgw"#location of pdfs in drive

    os.system(f"gdown --folder {folder_url} -O {base_download_path} --remaining-ok --quiet")# This call downloads the entire Google Drive folder to our local machine,
                                                                                            # ensuring all necessary articles are available for the indexing pipeline.

    final_folder = "./papers/papers" if (os.path.exists("./papers/papers") and any(f.endswith('.pdf') for f in os.listdir("./papers/papers"))) else "./papers"
    #incase path location i incorrect will allow to search in other locations for paper foldair
    pdf_files = [f for f in os.listdir(final_folder) if f.endswith(".pdf")]#get all pdf files into list

    for file_name in pdf_files:
        try:
            doc = fitz.open(os.path.join(final_folder, file_name))#open each pdf file into RAM
            for page_num, page in enumerate(doc):#enumirate allows us to get both page num and content
                raw_text = page.get_text("text", sort=True)
                if raw_text:
                    cleaned_text = clean_text(raw_text)#remove unwated chars
                    for chunk in chunk_text(cleaned_text):
                        rag_chunks.append({"file_name": file_name, "page": page_num + 1, "text": chunk}) # get chunks to be in Dictionary type that we can send to generator AI
        except Exception:
            pass#takes care of falude files

    if not rag_chunks:
        return "couldnt find chunks."

    documents = [chunk["text"] for chunk in rag_chunks]#list of all the chunks we found for later use

    vectorizer = TfidfVectorizer(stop_words="english")#ignore stop words from the english directory #noise filter # Initialize TF-IDF Vectorizer to convert text to numerical vectors
    tfidf_matrix = vectorizer.fit_transform(documents)# build directory from the document

    return f"RAG Ready! got  {len(rag_chunks) } chunks."
    #}
#initialize_rag_system()


In [ ]:
# ==========================================
# RAG SEARCH
# ==========================================
def gradio_rag_interface(user_text):
  #{
    try:
        if not user_text.strip():#if empty text
            return "Please enter a question."

        clean_query = clean_text(user_text)#get reads of unwated characters


        synonym_prompt = f"Generate 5 scientific synonyms or related keywords for this query: '{clean_query}'. Return ONLY the words separated by spaces, nothing else."
        expanded_words = call_gemini_direct(synonym_prompt)
        super_query = clean_query + " " + expanded_words
        query_vector = vectorizer.transform([super_query])#turn the question into vector we can work with for TF-IDF

        similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()#compare the question vector to all the vectors in the answer matrix to find similar answer

        top_indices = similarities.argsort()[::-1]#reverse the answers so that they will be ranked from best to worst #argsort sorts the answers as to be used in rankj
        relevant_chunks = []#saves the asked questions
        seen_files = set()#saves the filenames that we took answers too from allready so that later will allow to draw answer for diverce sorces

        for idx in top_indices:#go throu the chunks from most similar to lowers if we didnt take info from this source add it to the list of relevent_chunks and add it to seen fils so we wont look at it again
            file_name = rag_chunks[idx]['file_name']
            if file_name not in seen_files:
                relevant_chunks.append(rag_chunks[idx])
                seen_files.add(file_name)

            if len(relevant_chunks) >= 4:#stop looking after 4 sources
                break

        context_text = ""
        #From here we have the interface!!
        html = "<div style='padding-right: 15px;'><h2 style='color:#2563eb;'> Retrieved Sources</h2>"

        for i, chunk in enumerate(relevant_chunks):#just adds the number of article to the file print for clarafication #only print first 350 chars since it might be to big for the interface
            html += f"""
            <div style="padding:15px; margin-bottom:10px; border-radius:10px; border-left:5px solid #2563eb; background: var(--background-fill-secondary); color: var(--body-text-color);">
            <b style='color:#2563eb;'>Source {i+1}</b><br>📄 File: {chunk['file_name']}<br>📑 Page: {chunk['page']}<br><br>
            <span style='font-size:14.5px;'>{chunk['text'][:350]}...</span>
            </div>
            """
            context_text += f"\nDOCUMENT: {chunk['file_name']}\nPAGE: {chunk['page']}\nCONTENT:\n{chunk['text']}\n"#this is what we send to gemini

        html += "</div>"

        prompt = f"""
        Answer the question ONLY using the provided context.
        Cite filenames and page numbers.
        Do NOT use any outside knowledge under any circumstances.
        If the answer is not fully found in the context, say exactly: "The answer is not found in the provided context."

        CONTEXT:
        {context_text}

        QUESTION:
        {user_text}
        """

        answer = call_gemini_direct(prompt)

        html += f"""
        <hr><div style="padding:20px; border-radius:10px; border:1px solid var(--border-color-primary); background: var(--background-fill-secondary); color: var(--body-text-color); line-height:1.6;">
        <h2 style='color:#2563eb; margin-top:0;'> RAG Answer</h2>
        <span style='font-size:15px; white-space: pre-wrap;'>{answer}</span>
        </div>
        """

        return html

    except Exception as e:
        return f"<div style='color:#b91c1c; padding:20px;'>Error: {str(e)}</div>"

            #}
            #def gradio_rag_interface(user_text):

In [ ]:
# ==========================================
# UI
# ==========================================
custom_css = """
.gradio-container { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
#search-btn { background: linear-gradient(to right, #059669, #10b981) !important; border: none !important; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1); transition: all 0.2s ease-in-out; color: white !important;}
#search-btn:hover { transform: translateY(-2px); box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate"), css=custom_css, title="CropSurvive Academic Engine") as demo:

    gr.HTML("""
    <div style="padding: 35px 20px; background: linear-gradient(135deg, #064e3b 0%, #059669 100%); border-radius: 16px; color: white; text-align: center; box-shadow: 0 10px 25px -5px rgba(5, 150, 105, 0.4); margin-bottom: 30px; position: relative; overflow: hidden;">
        <div style="position: absolute; top: -15px; left: -15px; font-size: 120px; opacity: 0.05;">🌿</div>
        <h1 style='margin:0; font-size: 2.8em; font-weight: 800; letter-spacing: -0.5px; text-shadow: 0 2px 4px rgba(0,0,0,0.2);'> CropSurvive</h1>
        <p style='font-size: 1.2em; opacity: 0.9; margin-top: 8px; font-weight: 300;'>Academic Research Engine</p>
        <div style="margin-top: 15px; display: inline-block; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px; font-size: 0.9em; backdrop-filter: blur(5px); font-weight: 500;">
            TF-IDF Indexing &nbsp;&nbsp; Strict RAG Generation
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=4):

            input_box = gr.Textbox(
                label="Enter your research query",
                placeholder="e.g., What are the main components of the biological coating?",
                lines=3,
                elem_id="search-box",
                show_label=True
            )

        with gr.Column(scale=1, min_width=120):

            submit_btn = gr.Button("Search ", variant="primary", elem_id="search-btn")

            clear_btn = gr.Button("Clear", variant="secondary")



    gr.HTML("<hr style='margin-top: 20px; margin-bottom: 20px; border-top: 1px solid #e2e8f0;'/>")


    output_html = gr.HTML()


    submit_btn.click(fn=gradio_rag_interface, inputs=input_box, outputs=output_html)
    input_box.submit(fn=gradio_rag_interface, inputs=input_box, outputs=output_html)

    # Safe clear functionality
    clear_btn.click(fn=lambda: ("", ""), inputs=None, outputs=[input_box, output_html])

# ==========================================
# Launch
# ==========================================
print(initialize_rag_system())
discovery_model()
demo.launch(share=True)



example of possible questions:
1.   summarize the findings regarding the recommended storage temperature for the crops mentioned. - that one didnt find anwer last time i tried - because of too many requests i cant check anymore for now
2.   What are the main functions of alginate-based edible coatings in fresh-cut produce, and how do they affect the product's shelf life?
